## Basic unsupervised learning algorithms and working with text data

* 1\. Dimensionality reduction and data visualization
Apply the sklearn.decomposition.PCA and sklearn.manifold.TSNE dimensionality reduction methods to the data
Display the obtained results.

* 2\. Cluster analysis

    * 2.1\.  Using the k-means algorithm, perform image quantization (removal of visually redundant information)
        with 64, 32, 16, and 8 color levels. Choose any image you like.
        https://scikit-learn.org/stable/auto_examples/cluster/plot_color_quantization.html
    * 2.2\. Generate a synthetic dataset (points on a plane), for example, using sklearn.datasets.make_blobs.
        Choose the number of centers N (from 3 to 5) arbitrarily.
        Build silhouette plots for KMeans for N-1, N, and N+1 clusters and explain the results.
        Example: https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html


    * 2.3\. Generate a synthetic dataset in the form of a mixture of two Gaussians. For this, use the function:
        https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.random.multivariate_normal.html

(Apply it twice with different mean and covariance values), then combine the results into a single dataset.
Separate the mixture using the EM algorithm (sklearn.mixture.GaussianMixture).
Pay attention to the covariance_type parameter.
Using the weights_ and covariances_ attributes, reconstruct their values and compare them with the original ones.
Visualize the result.

* 3\. Text data processing

    * Load a labeled text dataset.
    * Perform data preprocessing (remove stop words, punctuation, and normalize the text).
    * Build a visualization of the most frequent words or n-grams in each class (word cloud)
    * Extract features (for example, using sklearn.feature_extraction.text.TfidfVectorizer or sklearn.decomposition.TruncatedSVD).
    * Perform text classification and evaluate its quality.

Text data for analysis can be taken from here:
https://lionbridge.ai/datasets/the-best-25-datasets-for-natural-language-processing/
or from any other source of your choice (if the data has many classes, it is enough to take 2–3 classes)

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from numpy.random import multivariate_normal
from skimage import data
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.utils import shuffle
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from wordcloud import WordCloud, STOPWORDS

### Load dataset

In [ ]:
df = pd.read_csv("bird.csv")
df = df.fillna(df.mean(numeric_only=True))
df = df.drop(columns=["id"])
feature_columns = [
    "huml",
    "humw",
    "ulnal",
    "ulnaw",
    "feml",
    "femw",
    "tibl",
    "tibw",
    "tarl",
    "tarw",
]

### Dimensionality reduction and visualization (PCA and TSNE)

In [ ]:
X = df[feature_columns]
pca = PCA(n_components=2).fit(X)
X_pca = pca.transform(X)
X_tsne = TSNE(n_components=2).fit_transform(X)

labels_pca = {
    str(i): f"PC {i + 1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
}
labels_tsne = {"0": "TS 1", "1": "TS 2"}
fig1 = px.scatter_matrix(
    X_pca, labels=labels_pca, dimensions=range(2), color=df["type"], title="PCA"
)
fig2 = px.scatter_matrix(
    X_tsne, labels=labels_tsne, dimensions=range(2), color=df["type"], title="TSNE"
)
fig1.show()
fig2.show()

### Cluster analysis

Using the k-means algorithm, quantize the image (removing visually redundant information)
with depths of 64, 32, 16, and 8 levels.
You can choose an image at random.


In [ ]:
image = data.chelsea()

In [ ]:
def color_quantization_k_means(
        image: np.ndarray,
        n_colors_list:list[int],
        sample_size:int=1000,
        random_state:int=0
)-> None:
    """Quantize an image using K-Means clustering."""
    # Convert image to float and normalize pixel values to [0, 1]
    image = np.array(image, dtype=np.float64) / 255.0

    # Image shape: height, width, channels
    height, width, channels = image.shape
    assert channels == 3, "Expected an RGB image with 3 channels."

    # Flatten image into a 2D array: each row is one pixel
    pixels = image.reshape(-1, channels)

    # Train one K-Means model per palette size and plot the results
    n_results = len(n_colors_list)
    fig, axes = plt.subplots(1, n_results + 1, figsize=(5 * (n_results + 1), 6))

    # Show original image
    axes[0].imshow(image)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    for i, n_colors in enumerate(n_colors_list, start=1):
        # Use a subset of pixels to train K-Means for speed
        sample = shuffle(pixels, random_state=random_state)[:sample_size]
        kmeans = KMeans(n_clusters=n_colors, random_state=random_state).fit(sample)

        # Assign every pixel to the nearest cluster center
        labels = kmeans.predict(pixels)

        # Replace each pixel by its cluster center
        quantized_pixels = kmeans.cluster_centers_[labels]
        quantized_image = quantized_pixels.reshape(height, width, channels)

        axes[i].imshow(quantized_image)
        axes[i].set_title(f"{n_colors} colors")
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
color_quantization_k_means(image, n_colors_list=[8, 16, 32,64])

Generate a synthetic dataset (points on a plane), for example, using
sklearn.datasets.make_blobs. Choose an arbitrary number of centers N (from 3 to 5).
Plot silhouette plots for Kmeans (for N-1, N, N+1 clusters), and explain the results.

In [ ]:
X, y = make_blobs(
    n_samples=500,
    n_features=2,
    centers=4,
    cluster_std=1,
    center_box=(-10.0, 10.0),
    shuffle=True,
    random_state=1,
)
plt.figure(figsize=(10, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.5, color="blue")
plt.show()
plt.close()

In [ ]:
range_n_clusters = [4, 6, 7, 8]

for n_clusters in range_n_clusters:
    # Create a figure with two panels:
    # 1) silhouette plot
    # 2) clustered data visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    # Silhouette plot limits
    ax1.set_xlim([-0.1, 1.0])
    ax1.set_ylim([0, len(X) + (n_clusters + 1) * 10])

    # Fit KMeans and obtain cluster labels
    clusterer = KMeans(n_clusters=n_clusters, random_state=10)
    cluster_labels = clusterer.fit_predict(X)

    # Mean silhouette score for the whole clustering
    silhouette_avg = silhouette_score(X, cluster_labels)
    print(f"For n_clusters = {n_clusters} the average silhouette_score is: {silhouette_avg}")

    # Silhouette values for each sample
    sample_silhouette_values = silhouette_samples(X, cluster_labels)

    # Build the silhouette plot cluster by cluster
    y_lower = 10
    for cluster_id in range(n_clusters):
        # Take silhouette values for the current cluster and sort them
        cluster_values = sample_silhouette_values[cluster_labels == cluster_id]
        cluster_values.sort()

        size_cluster = cluster_values.shape[0]
        y_upper = y_lower + size_cluster

        color = cm.nipy_spectral(float(cluster_id) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            cluster_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )

        # Label each cluster on the plot
        ax1.text(-0.05, y_lower + 0.5 * size_cluster, str(cluster_id))

        y_lower = y_upper + 10

    ax1.set_title("Silhouette plot for KMeans clustering")
    ax1.set_xlabel("Silhouette coefficient values")
    ax1.set_ylabel("Cluster label")
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--", label="Average silhouette score")
    ax1.set_yticks([])
    ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

    # Visualize the clustered samples
    colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
    ax2.scatter(
        X[:, 0], X[:, 1],
        marker=".",
        s=30,
        lw=0,
        alpha=0.7,
        c=colors,
        edgecolor="k",
    )

    # Plot cluster centers
    centers = clusterer.cluster_centers_
    ax2.scatter(
        centers[:, 0],
        centers[:, 1],
        marker="o",
        c="white",
        alpha=1,
        s=200,
        edgecolor="k",
    )

    # Annotate cluster centers with their indices
    for cluster_id, center in enumerate(centers):
        ax2.scatter(center[0], center[1], marker=f"${cluster_id}$", alpha=1, s=50, edgecolor="k")

    ax2.set_title("Clustered data visualization")
    ax2.set_xlabel("Feature 1")
    ax2.set_ylabel("Feature 2")

    fig.suptitle(
        f"Silhouette analysis for KMeans clustering with n_clusters = {n_clusters}",
        fontsize=14,
        fontweight="bold",
    )

    plt.tight_layout()
    plt.show()
    plt.close()

Generate a synthetic dataset in the form of a mixture of two Gaussians.

To do this, use the function:
https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.random.multivariate_normal.html

(Apply it twice with different mean and covariance values),
then combine the results into a single dataset.

Separate the mixture using the EM algorithm (sklearn.mixture.GaussianMixture).

Pay attention to the covariance_type parameter.

Using the weights_ and covariances_ attributes, recover their values
and compare them with the original ones.

Visualize the result.

In [ ]:
mean1 = (2, 3)
cov1 = [[1, 0], [0, 1]]
X1 = multivariate_normal(mean1, cov1, 200)

mean2 = (5, 6)
cov2 = [[2, 0], [0, 2]]
X2 = multivariate_normal(mean2, cov2, 200)
X = np.concatenate([X1, X2])

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], color="blue")
plt.show()
plt.close()

In [ ]:
clf = GaussianMixture(n_components=2, covariance_type="full")
clf.fit(X)
predicted = clf.predict(X)
plt.figure(figsize=(5, 5))
for k in range(0, 2):
    data = X[predicted == k]
    plt.scatter(data[:, 0], data[:, 1], c=["red", "green"][k])
plt.show()
plt.close()

### Text data process

In [ ]:
twenty_train = fetch_20newsgroups(subset="train", shuffle=True)

count_vect = CountVectorizer(
    token_pattern=r"(?u)\b[a-zA-Z]\w+\b", ngram_range=(1, 2), stop_words="english"
)
X_train_counts = count_vect.fit_transform(twenty_train.data)
X_train_counts.shape

In [ ]:
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
X_train_tfidf.shape

In [ ]:
clf = MultinomialNB().fit(X_train_tfidf, twenty_train.target)
clf.score(X_train_counts, twenty_train.target)

In [ ]:
wc = WordCloud(width=1000, height=1000, stopwords=STOPWORDS)
text = " ".join(count_vect.get_feature_names_out(twenty_train.target_names))
embed_code = wc.generate(text=text)
plt.figure(figsize=(20, 45), dpi=100)
plt.imshow(embed_code)